# Microsoft Foundry - AI Agent Telemetry Notebook Guide

This notebook demonstrates how to create and interact with an AI agent using the **Azure AI Projects SDK**. The agent is configured as a storytelling agent that crafts engaging stories based on user prompts. It also enables **OpenTelemetry tracing** to send agent telemetry to Application Insights and the `AppDependancies` table in Log Analytics.

## 0. Create/Re-use Virtual Environment & Register Kernel
- Create (or re-use) a Python virtual environment to isolate dependencies for this project, then install ipykernel and register it as a notebook kernel.
- Once this cell is completed, be sure to 'Select Kernel' and choose your new .venv environment.

In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (safe to run repeatedly)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Resolve the venv Python path per OS, then use "python -m pip" for reliability
venv_python = (
    os.path.join(venv_dir, "Scripts", "python.exe")
    if os.name == "nt"
    else os.path.join(venv_dir, "bin", "python")
)

# Install / upgrade ipykernel inside the venv
subprocess.check_call([venv_python, "-m", "pip", "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
subprocess.check_call([
    venv_python,
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    "ai-agent-demo",
    "--display-name",
    "AI Agent Demo (.venv)",
])

print(f"Virtual environment created or reused at {venv_dir}")
print("Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

## 1. Install Dependencies

Install the required Python packages:
- **azure-ai-projects** — SDK for working with Azure AI Foundry projects and agents
- **azure-identity** — Provides `DefaultAzureCredential` for seamless Azure authentication
- **azure-monitor-opentelemetry** — All-in-one Azure Monitor OpenTelemetry distro (configures exporters, tracer providers, etc.)
- **opentelemetry-sdk** — Core OpenTelemetry SDK for instrumentation and tracing
- **azure-core-tracing-opentelemetry** — Azure SDK tracing plugin for OpenTelemetry

In [ ]:
import json
import subprocess
import sys

outdated = subprocess.check_output(
    [sys.executable, "-m", "pip", "list", "--outdated", "--format=json"],
    text=True,
)
if any(pkg["name"].lower() == "pip" for pkg in json.loads(outdated)):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Keep Azure Monitor exporter current to avoid OpenTelemetry import mismatches.
%pip --disable-pip-version-check install --upgrade --pre azure-monitor-opentelemetry-exporter
%pip --disable-pip-version-check install --pre "azure-ai-projects>=2.0.0b4"
%pip --disable-pip-version-check install azure-identity azure-monitor-opentelemetry azure-core-tracing-opentelemetry

## 2. Import Libraries

Import the necessary classes from the Azure SDKs:
- `DefaultAzureCredential` — Automatically picks up credentials from your environment (Azure CLI, managed identity, etc.)
- `AIProjectClient` — Client for interacting with your Azure AI Foundry project
- `PromptAgentDefinition` — Defines the agent's model and behavior instructions
- `AIProjectInstrumentor` — Instruments all azure-ai-projects and OpenAI SDK operations for tracing (imported in Section 3.5)
- `configure_azure_monitor` — One-liner to set up Azure Monitor as the OpenTelemetry export backend

In [ ]:
import os, sys, platform

# Set resource detector override BEFORE importing azure-monitor-opentelemetry
# to ensure it takes effect during module initialization
if platform.system() == "Darwin":
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"
    print(f"⚙️  macOS detected — disabled Azure IMDS resource detector to avoid local hang")

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.telemetry import AIProjectInstrumentor

print("✅ All libraries imported successfully")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

## 3. Configure the Project Client

Set the Azure AI Foundry project endpoint and create an authenticated `AIProjectClient` instance. Update the `user_endpoint` to match your own project URL.

In [ ]:
import logging
import os
import subprocess

user_endpoint = "https://zolab1702-ai-foundry.services.ai.azure.com/api/projects/zolab1702-ai-fndry-proj"

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=user_endpoint,
    credential=credential,
)

print("✅ AIProjectClient configured")
print(f"   🌐 Endpoint: {user_endpoint}")
print("   🔐 Auth:     DefaultAzureCredential")

# Optional: Return the signed-in user for certain credential types to provide more transparency about which identity is being used.
# This is best-effort and may not succeed for all credential types, but can be helpful for local development scenarios where multiple credentials may be available.
def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None

def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        az_exe = "az.cmd" if os.name == "nt" else "az"
        return _run_command([az_exe, "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None

# Optional credential-source probe for transparency
try:
    identity_logger = logging.getLogger("azure.identity")
    if not identity_logger.handlers:
        identity_logger.addHandler(logging.StreamHandler())
    identity_logger.setLevel(logging.INFO)

    _ = credential.get_token("https://management.azure.com/.default")
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        print(f"   🔐 Credential used: {credential_name}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            print(f"   👤 Signed-in account: {identity_hint}")
        else:
            print("   👤 Signed-in account: not available for this credential type")
    else:
        print("   Credential used: check azure.identity INFO logs above")
except Exception as ex:
    print(f"   Credential probe skipped: {type(ex).__name__}: {ex}")

## 3.5 Enable Telemetry

Configure tracing per the [azure-ai-projects SDK tracing guide](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects#tracing):
1. **`configure_azure_monitor`** — Sets up the full OpenTelemetry pipeline (TracerProvider, exporter, span processors) to send traces to Application Insights
2. **`AIProjectInstrumentor`** — Instruments all `azure-ai-projects` SDK operations (agent create/version, list, etc.) and automatically instruments OpenAI responses/conversations operations

> **Note:** `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` must be set **before** calling `instrument()`. Content recording is controlled by `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT`.

In [ ]:

from opentelemetry import trace

# Must be set BEFORE importing/calling monitor + instrument()
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

# Enable end-to-end correlation between client and service spans
os.environ["AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION"] = "true"
os.environ["AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE"] = "true"

# Get the Application Insights connection string from the Foundry project
application_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# Note: The connection string is required to configure the OpenTelemetry exporter, but the underlying Azure Monitor 
# exporter will also look for it in the AZURE_MONITOR_CONNECTION_STRING environment variable as a fallback, 
# so it's not strictly necessary to pass it here if it's already set in the environment.
from azure.monitor.opentelemetry import configure_azure_monitor

# Configure Azure Monitor as the tracing backend (one-liner per SDK docs)
configure_azure_monitor(connection_string=application_insights_connection_string)

# Instrument azure-ai-projects SDK + OpenAI responses/conversations automatically
AIProjectInstrumentor().instrument(enable_content_recording=True)

tracer = trace.get_tracer(__name__)

print(f"Tracing enabled → Application Insights (connection string retrieved from project)")

## 4. Create the Agent

Define and create a versioned agent. Replace `<your-agent-name>` and `<your-model-deployment-name>` with your actual values.

- `create_version` will create a new agent or bump the version if the parameters have changed.
- The agent is given a **storytelling** persona via the `instructions` field.

In [ ]:
agent_name = "ZoDEfendersAgent-1702"
model_name = "gpt-4.1-mini"
agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context."
)

# Wrap agent creation in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-creation") as span:
    span.set_attribute("agent.name", agent_name)
    span.set_attribute("gen_ai.request.model", model_name)

    # Creates an agent, bumps the agent version if parameters have changed
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=agent_instructions,
        ),
    )
    span.set_attribute("agent.version", agent.version)
    span.set_attribute("agent.id", agent.id)

print(f"Agent created: {agent.name} (id: {agent.id}, version: {agent.version})")

## 5. Query the Agent

Use the OpenAI-compatible client from the project to send a prompt to the agent and retrieve a response. The agent is referenced by name using `agent_reference`. Each response is appended to `stories.json` as a new object in the array. Access individual stories via `stories[0]`, `stories[1]`, etc.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

openai_client = project_client.get_openai_client()

user_prompt = ( 
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, Germany. "
    "He is a man who moonlights as a cybernetic superhero called 'Die DEfender', protector of the digital realm. "
    "Include a plot twist in the last sentence. Always refer to the two characters by their full names."
)

# Wrap the agent query in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-query") as span:
    span.set_attribute("agent.name", agent.name)
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.prompt", user_prompt)

    # Reference the agent to get a response
    response = openai_client.responses.create(
        input=[{"role": "user", "content": user_prompt}],
        extra_body={"agent_reference": {"name": agent.name, "id": agent.id, "type": "agent_reference"}},
    )

    span.set_attribute("gen_ai.completion", response.output_text[:500])
    span.set_attribute("gen_ai.response.id", response.id)

print(f"Response output: {response.output_text}")

def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
story_id = append_story(
    stories_file,
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": agent.name,
        "model": model_name,
        "prompt": user_prompt,
        "story": response.output_text,
    },
)

print(f"\nStory #{story_id} saved to {stories_file}")

## 6. Validate Traces in Log Analytics (Security Subscription)

Use this section to validate that telemetry from the notebook is landing in the Log Analytics workspace hosted in the Security subscription.

- Workspace resource ID: `/subscriptions/<subcription-id>/resourceGroups/Sentinel/providers/Microsoft.OperationalInsights/workspaces/<Log-Analytics-Workspace-Name>`
- The Azure CLI extension command `az monitor log-analytics query` may fail in some environments due to extension/runtime mismatch.
- The code cell below uses `az rest` against `api.loganalytics.io`, which is a reliable fallback.

The first query gives an end-to-end joined view of dependencies and traces by `OperationId`.
The second query gives a compact runs-only trend with call volume, failures, and latency percentiles.

In [ ]:
import json
import urllib.error
import urllib.request

from azure.identity import DefaultAzureCredential

workspace_customer_id = "7e9298ab-22e6-4a82-a53e-c5ed7faee977"

# End-to-end view including requested fields: agent name, model, system message, host, location
kql_end_to_end = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
| where Name in ("agent-query", "responses") or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    system_message = coalesce(
        extract(@'"role"\s*:\s*"system".*?"content"\s*:\s*"([^\"]+)"', 1, tostring(Properties["gen_ai.input.messages"])),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    )
| project
    TimeGenerated,
    Name,
    agent_name,
    model_used,
    system_message,
    computer,
    location,
    Success,
    DurationMs,
    OperationId,
    Prompt = tostring(Properties["gen_ai.prompt"]),
    Completion = tostring(Properties["gen_ai.completion"])
| order by TimeGenerated desc
| take 50
""".strip()

# Runs-only trend grouped by agent/model/host/location
kql_runs_trend = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
| where Name in ("agent-query", "responses") or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    )
| summarize
    Calls = count(),
    Failures = countif(Success == false),
    AvgDurationMs = avg(DurationMs),
    P95DurationMs = percentile(DurationMs, 95)
    by bin(TimeGenerated, 15m), agent_name, model_used, computer, location
| order by TimeGenerated desc
""".strip()

def query_log_analytics(customer_id: str, query: str) -> dict:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://api.loganalytics.io/.default").token

    body = json.dumps({"query": query}).encode("utf-8")
    request = urllib.request.Request(
        url=f"https://api.loganalytics.io/v1/workspaces/{customer_id}/query",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token}",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as ex:
        error_body = ex.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Log Analytics query failed ({ex.code}): {error_body}") from ex

print("Running end-to-end validation query...")
end_to_end_result = query_log_analytics(workspace_customer_id, kql_end_to_end)
print(json.dumps(end_to_end_result, indent=2)[:4000])

print("\nRunning runs-only trend query...")
runs_trend_result = query_log_analytics(workspace_customer_id, kql_runs_trend)
print(json.dumps(runs_trend_result, indent=2)[:4000])